In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

In [2]:
project_dir = os.path.dirname(os.getcwd())
games = pd.read_csv(os.path.join(project_dir,"data/final_chess.csv"), parse_dates=["Date"])

In [3]:
X = games.drop(columns=[
    "White", "Black", "Result", "Site", "ECO",
    # White/Black color counts
    'WasBWinRate', 'WasBLossRate', 'WasBDrawRate', 'BasWWinRate', 'BasWLossRate', 'BasWDrawRate',
    "WasBGames", "WasBWin", "WasBLoss", "WasBDraw",
    "BasWGames", "BasWWin", "BasWLoss", "BasWDraw",
])
y1 = games["Result"].map({1: 1, 0.5: 0, 0:1}) # draw or decisive
y2 = games["Result"].map({1: 2, 0.5: 1, 0: 0}) #sklearn only considers integers

"""
0 = Black win
1 = Draw
2 = White win
"""

'\n0 = Black win\n1 = Draw\n2 = White win\n'

In [4]:
Xtrain = X[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2023)].drop(columns="Date")
y1train = y1[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2023)].drop(columns="Date")
y2train = y2[(X["Date"].dt.year >= 2016) & (X["Date"].dt.year <= 2023)].drop(columns="Date")
Xval   = X[(X["Date"].dt.year == 2025) | (X["Date"].dt.year == 2024)].drop(columns="Date")
y1val   = y1[(X["Date"].dt.year == 2025) | (X["Date"].dt.year == 2024)].drop(columns="Date")
y2val   = y2[(X["Date"].dt.year == 2025) | (X["Date"].dt.year == 2024)].drop(columns="Date")
Xtest  = X[X["Date"].dt.year == 2026].drop(columns="Date")
y1test  = y1[X["Date"].dt.year == 2026].drop(columns="Date")
y2test  = y2[X["Date"].dt.year == 2026].drop(columns="Date")

In [5]:
from xgboost import XGBClassifier

In [11]:
xgbc = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",

    colsample_bytree=0.5158019945643415,
    gamma=0.18576353181073108,
    learning_rate=0.0238211139654102,
    max_depth=6,
    min_child_weight=7,
    n_estimators=733,
    reg_alpha=0.2155168238099609,
    reg_lambda=1.285808534044257,
    subsample=0.6576010179563144,

    random_state=67,
    tree_method="hist",
    n_jobs=-1
)

In [ ]:
#xgbc.fit(Xtrain, y2train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.5158019945643415
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.dat

In [6]:
model_path = os.path.join(project_dir, "results/models/xgbc_61_base.joblib")
xgbc = joblib.load(model_path)

In [7]:
ypred = xgbc.predict(Xval)

In [8]:
confusion_matrix(y2val, ypred)

array([[207259,  15057,  85821],
       [ 52914,  36861,  63198],
       [ 80718,  16172, 264452]])

In [9]:
classification_report(y2val, ypred).splitlines()

['              precision    recall  f1-score   support',
 '',
 '           0       0.61      0.67      0.64    308137',
 '           1       0.54      0.24      0.33    152973',
 '           2       0.64      0.73      0.68    361342',
 '',
 '    accuracy                           0.62    822452',
 '   macro avg       0.60      0.55      0.55    822452',
 'weighted avg       0.61      0.62      0.60    822452']

In [10]:
prob_val = xgbc.predict_proba(Xval)

In [11]:
actual_draw = (y2val == 1)
roc_auc_score(actual_draw, prob_val[:,1])

0.7538191521906906

In [12]:
thresholds = np.arange(0.20, 0.51, 0.01)

results = []

for threshold in thresholds:

    ypred = np.where(
        prob_val[:,1] >= threshold,
        1,
        np.where(
            prob_val[:, 0] > prob_val[:, 2],
            0,
            2
        )
    )

    report = classification_report(
        y2val,
        ypred,
        output_dict=True
    )

    results.append({
        "threshold": threshold,
        "accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"],
        "black_f1": report["0"]["f1-score"],
        "draw_f1": report["1"]["f1-score"],
        "white_f1": report["2"]["f1-score"],
    })

threshold_results = pd.DataFrame(results)

In [13]:
threshold_results.sort_values(
    "macro_f1",
    ascending=False
).head(10)

,threshold,accuracy,macro_f1,black_f1,draw_f1,white_f1
9,0.29,0.609694,0.578221,0.624042,0.443823,0.666797
7,0.27,0.604530,0.578211,0.618213,0.455653,0.660767
8,0.28,0.607097,0.578209,0.621291,0.449593,0.663744
10,0.30,0.611720,0.577549,0.626337,0.437035,0.669276
6,0.26,0.601228,0.577199,0.614686,0.459648,0.657262
11,0.31,0.613294,0.576239,0.628352,0.428947,0.671418
5,0.25,0.597701,0.575906,0.611004,0.463377,0.653337
12,0.32,0.614782,0.574729,0.630314,0.420478,0.673396
4,0.24,0.593405,0.573635,0.606433,0.465446,0.649026
13,0.33,0.615941,0.572775,0.631885,0.411323,0.675116


In [14]:
threshold_results.sort_values(
    "draw_f1",
    ascending=False
).head(10)

,threshold,accuracy,macro_f1,black_f1,draw_f1,white_f1
1,0.21,0.578335,0.564285,0.590982,0.468866,0.633007
2,0.22,0.583678,0.567799,0.596464,0.468200,0.638733
0,0.20,0.572071,0.559720,0.584797,0.467935,0.626429
3,0.23,0.588769,0.570992,0.601586,0.467254,0.644135
4,0.24,0.593405,0.573635,0.606433,0.465446,0.649026
5,0.25,0.597701,0.575906,0.611004,0.463377,0.653337
6,0.26,0.601228,0.577199,0.614686,0.459648,0.657262
7,0.27,0.604530,0.578211,0.618213,0.455653,0.660767
8,0.28,0.607097,0.578209,0.621291,0.449593,0.663744
9,0.29,0.609694,0.578221,0.624042,0.443823,0.666797


In [15]:
threshold_results.sort_values(
    "accuracy",
    ascending=False
).head(10)

,threshold,accuracy,macro_f1,black_f1,draw_f1,white_f1
17,0.37,0.618051,0.561404,0.636440,0.368119,0.679653
18,0.38,0.617877,0.557478,0.636988,0.355151,0.680296
16,0.36,0.617804,0.564731,0.635621,0.379911,0.678662
19,0.39,0.617702,0.553644,0.637599,0.342535,0.680798
15,0.35,0.617465,0.567852,0.634621,0.391255,0.677682
20,0.40,0.617407,0.549469,0.638065,0.329101,0.681242
21,0.41,0.617241,0.545682,0.638592,0.316769,0.681686
22,0.42,0.616770,0.541433,0.638843,0.303482,0.681973
14,0.34,0.616708,0.570313,0.633216,0.401223,0.676501
23,0.43,0.616229,0.537045,0.639002,0.289907,0.682224


In [16]:
best_threshold = 0.29

In [17]:
ypred_xgb = np.where(
    prob_val[:,1] >= best_threshold,
    1,
    np.where(
        prob_val[:, 0] > prob_val[:, 2],
        0,
        2
    )
)

In [18]:
print(confusion_matrix(y2val, ypred_xgb))
print(classification_report(y2val, ypred_xgb))

[[190916  38875  78346]
 [ 39194  67446  46333]
 [ 73622  44638 243082]]
              precision    recall  f1-score   support

           0       0.63      0.62      0.62    308137
           1       0.45      0.44      0.44    152973
           2       0.66      0.67      0.67    361342

    accuracy                           0.61    822452
   macro avg       0.58      0.58      0.58    822452
weighted avg       0.61      0.61      0.61    822452



In [ ]:
#joblib.dump(xgbc, os.path.join(project_dir, "results/models/xgbc_61_base.joblib"))

['/Users/vallurileelasaikrishna/chess/results/models/xgbc_61_base.joblib']

### Final test between rfc and xgbc

In [19]:
model_path = os.path.join(project_dir, "results/models/rfc_62_base.joblib")
rfc = joblib.load(model_path)

In [20]:
xgb_prob = xgbc.predict_proba(Xtest)
xgb_draw_prob = xgb_prob[:, 1]

xgb_test = np.where(
    xgb_draw_prob >= 0.29,
    1,
    np.where(
        xgb_prob[:, 0] > xgb_prob[:, 2],
        0,
        2
    )
)

In [24]:
rfc_prob = rfc.predict_proba(Xtest)
rfc_draw_prob = rfc_prob[:, 1]

rfc_test = np.where(
    rfc_draw_prob >= 0.37,
    1,
    np.where(
        rfc_prob[:, 0] > rfc_prob[:, 2],
        0,
        2
    )
)

In [25]:
print(confusion_matrix(y2test, xgb_test))
print(classification_report(y2test, xgb_test))

[[33178  6410 12954]
 [ 6259 11597  7622]
 [12121  7477 41870]]
              precision    recall  f1-score   support

           0       0.64      0.63      0.64     52542
           1       0.46      0.46      0.46     25478
           2       0.67      0.68      0.68     61468

    accuracy                           0.62    139488
   macro avg       0.59      0.59      0.59    139488
weighted avg       0.62      0.62      0.62    139488



In [26]:
print(confusion_matrix(y2test, rfc_test))
print(classification_report(y2test, rfc_test))

[[32999  6726 12817]
 [ 6240 11816  7422]
 [12348  7779 41341]]
              precision    recall  f1-score   support

           0       0.64      0.63      0.63     52542
           1       0.45      0.46      0.46     25478
           2       0.67      0.67      0.67     61468

    accuracy                           0.62    139488
   macro avg       0.59      0.59      0.59    139488
weighted avg       0.62      0.62      0.62    139488



Therefore, from the above results it looks like both are giving the similar results.